# BA-GSAC: baselines added in review

Code for: Haklidir, M. (2026). When Does Adaptive Guidance Help? Belief-Aware Privileged Distillation for Autonomous Driving Under Partial Observability. arXiv:2605.26155, accepted at the CVPR 2026 Workshop on Autonomous Driving (WAD).

This notebook runs the baselines added in review: a fixed λ = 0.01 baseline (3 seeds, 3 POMDP levels), a fixed λ = 0.5 baseline (3 seeds, severe), a linear-decay λ schedule (3 seeds, severe), SAC with an observation history of K = 3 (3 seeds, 3 POMDP levels), multi-seed runs for the mild and moderate levels (BA-GSAC, GSAC, SAC), and collision-rate tracking for all runs. These runs are the source of the severe-POMDP comparison in the paper, where the linear decay schedule performs best.

Estimated runtime: 15 to 18 hours on a Colab T4 GPU. Runtime > Run all; results are saved as JSON after each experiment block.

## 1. Environment Setup

In [ ]:
%%capture
!pip install highway-env gymnasium torch numpy matplotlib pandas seaborn tqdm

## 2. Imports and Configuration

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import highway_env
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import seaborn as sns
sns.set_style("whitegrid")
import pandas as pd
from copy import deepcopy
from collections import deque
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Highway-Env version: {highway_env.__version__}")

## 3. Highway-Env configuration for the POMDP experiments

Highway-Env is configured with continuous actions (steering and acceleration, matching GSAC's continuous action space) and kinematics observations with configurable noise and occlusion. Two scenarios are defined: highway (lane changing) and intersection (negotiation).

In [ ]:
# Environment Configurations

HIGHWAY_CONFIG = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,
        "features": ["presence", "x", "y", "vx", "vy"],
        "features_range": {"x": [-100, 100], "y": [-100, 100],
                           "vx": [-20, 20], "vy": [-20, 20]},
        "absolute": False,
        "normalize": True,
        "flatten": True,
        "order": "sorted",
    },
    "action": {"type": "ContinuousAction"},
    "lanes_count": 3,
    "vehicles_count": 10,
    "duration": 40,
    "collision_reward": -1.0,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "reward_speed_range": [20, 30],
    "policy_frequency": 5,
    "simulation_frequency": 15,
}

INTERSECTION_CONFIG = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 5,
        "features": ["presence", "x", "y", "vx", "vy"],
        "features_range": {"x": [-100, 100], "y": [-100, 100],
                           "vx": [-20, 20], "vy": [-20, 20]},
        "absolute": False,
        "normalize": True,
        "flatten": True,
        "observe_intentions": False,
    },
    "action": {"type": "ContinuousAction"},
    "duration": 13,
    "initial_vehicle_count": 8,
    "spawn_probability": 0.6,
    "collision_reward": -5.0,
    "normalize_reward": False,
}


def make_env(env_id="highway-v0", config=None):
    """Create and configure a highway-env environment."""
    cfg = config or HIGHWAY_CONFIG
    env = gym.make(env_id, config=cfg)
    return env


# Quick sanity check
env = make_env("highway-v0", HIGHWAY_CONFIG)
obs, info = env.reset()
print(f"Highway env created")
print(f"   Observation shape: {obs.shape}")
print(f"   Action space: {env.action_space}")
print(f"   Obs sample: {obs[:5]}")
env.close()

## 4. POMDP wrapper

The wrapper creates partial observability through Gaussian noise on the kinematic features, random masking of some vehicles (sensor limitations) and an optional observation delay. It keeps both the full state, for the guiding actor, and the partial observation, for the control actor, as in the Guided SAC setting.

In [ ]:
# POMDP Wrapper for Highway-Env

class POMDPWrapper(gym.Wrapper):
    """
    Wraps Highway-Env to create a POMDP setting.
    All observations are flattened to 1D for consistent tensor shapes.
    """

    PRESETS = {
        "mild":     {"noise_std": 0.02, "occlusion_prob": 0.10},
        "moderate": {"noise_std": 0.05, "occlusion_prob": 0.25},
        "severe":   {"noise_std": 0.10, "occlusion_prob": 0.50},
        "full":     {"noise_std": 0.00, "occlusion_prob": 0.00},
    }

    def __init__(self, env, pomdp_level="moderate"):
        super().__init__(env)
        preset = self.PRESETS.get(pomdp_level, self.PRESETS["moderate"])
        self.noise_std = preset["noise_std"]
        self.occlusion_prob = preset["occlusion_prob"]
        self.pomdp_level = pomdp_level
        self._full_obs = None
        # Compute flat obs dim
        self.flat_obs_dim = int(np.prod(env.observation_space.shape))

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        flat = obs.flatten().astype(np.float32)
        self._full_obs = flat.copy()
        partial = self._make_partial(flat)
        info["full_obs"] = self._full_obs.copy()
        return partial, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        flat = obs.flatten().astype(np.float32)
        self._full_obs = flat.copy()
        partial = self._make_partial(flat)
        info["full_obs"] = self._full_obs.copy()
        return partial, reward, terminated, truncated, info

    def get_full_state(self):
        return self._full_obs.copy() if self._full_obs is not None else None

    def _make_partial(self, obs):
        partial = obs.copy()
        if self.noise_std > 0:
            noise = np.random.normal(0, self.noise_std, size=partial.shape).astype(np.float32)
            partial = partial + noise
        # Occlusion: zero out vehicle groups (5 features each)
        if self.occlusion_prob > 0:
            n_features = 5
            n_vehicles = len(partial) // n_features
            for v in range(1, n_vehicles):  # skip ego
                if np.random.random() < self.occlusion_prob:
                    start = v * n_features
                    end = min(start + n_features, len(partial))
                    partial[start:end] = 0.0
        return np.clip(partial, -1.0, 1.0).astype(np.float32)


# Test POMDP wrapper
env = POMDPWrapper(make_env("highway-v0", HIGHWAY_CONFIG), pomdp_level="moderate")
partial_obs, info = env.reset()
full_obs = info["full_obs"]
print(f"POMDP Wrapper active (level: moderate)")
print(f"   Flat obs dim: {env.flat_obs_dim}")
print(f"   Full obs shape: {full_obs.shape}, Partial obs shape: {partial_obs.shape}")
print(f"   Difference (noise): {np.abs(full_obs - partial_obs).mean():.4f}")
env.close()

## 5. Guided SAC base components

The components of Guided SAC (Haklıdır and Temeltaş, IEEE Access 2021): the guiding actor π_g (full-state policy, used only in training), the control actor π_c (partial-observation policy, deployed at test time), the twin Q-network critic, the distillation network that transfers knowledge from the guiding to the control actor, and the observation history that gives the control actor temporal context.

In [ ]:
# Guided SAC Base Components

class GaussianPolicy(nn.Module):
    LOG_STD_MIN, LOG_STD_MAX = -20, 2

    def __init__(self, input_dim, action_dim, hidden_dim, action_scale=1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim, action_dim)
        self.log_std_head = nn.Linear(hidden_dim, action_dim)
        self.action_scale = action_scale

    def forward(self, x):
        h = self.net(x)
        mu = self.mu_head(h)
        log_std = self.log_std_head(h).clamp(self.LOG_STD_MIN, self.LOG_STD_MAX)
        return mu, log_std

    def sample(self, x):
        mu, log_std = self.forward(x)
        std = log_std.exp()
        dist = torch.distributions.Normal(mu, std)
        x_t = dist.rsample()
        action = torch.tanh(x_t) * self.action_scale
        log_prob = dist.log_prob(x_t) - torch.log(
            self.action_scale * (1 - torch.tanh(x_t).pow(2)) + 1e-6)
        log_prob = log_prob.sum(-1, keepdim=True)
        det_action = torch.tanh(mu) * self.action_scale
        return action, log_prob, det_action


class TwinQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super().__init__()
        dim = state_dim + action_dim
        self.q1 = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1))
        self.q2 = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1))

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        return self.q1(x), self.q2(x)


class DistillationDNN(nn.Module):
    def __init__(self, input_dim, action_dim, hidden_dim, action_scale=1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, action_dim), nn.Tanh())
        self.action_scale = action_scale

    def forward(self, x):
        return self.net(x) * self.action_scale


class ObservationHistory:
    """Fixed-length history of FLAT partial observations."""
    def __init__(self, obs_dim, history_len=3):
        self.obs_dim = obs_dim
        self.history_len = history_len
        self.total_dim = obs_dim * history_len
        self.buffer = deque(maxlen=history_len)

    def reset(self, initial_obs):
        flat = initial_obs.flatten().astype(np.float32)
        self.buffer.clear()
        for _ in range(self.history_len):
            self.buffer.append(flat.copy())

    def push(self, obs):
        flat = obs.flatten().astype(np.float32)
        self.buffer.append(flat)

    def get(self):
        """Return flattened concatenation of history. Shape: (obs_dim * history_len,)"""
        return np.concatenate(list(self.buffer), axis=0).astype(np.float32)


class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.buffer = []
        self.pos = 0

    def push(self, full_state, partial_hist, action, reward,
             next_full, next_hist, done):
        data = (full_state.flatten(), partial_hist.flatten(),
                np.array(action).flatten(), float(reward),
                next_full.flatten(), next_hist.flatten(), float(done))
        if len(self.buffer) < self.capacity:
            self.buffer.append(data)
        else:
            self.buffer[self.pos] = data
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.randint(0, len(self.buffer), batch_size)
        batch = [self.buffer[i] for i in indices]
        fs, ph, a, r, nfs, nph, d = zip(*batch)
        return {
            "full_states": np.array(fs, dtype=np.float32),
            "partial_hists": np.array(ph, dtype=np.float32),
            "actions": np.array(a, dtype=np.float32),
            "rewards": np.array(r, dtype=np.float32).reshape(-1, 1),
            "next_full_states": np.array(nfs, dtype=np.float32),
            "next_partial_hists": np.array(nph, dtype=np.float32),
            "dones": np.array(d, dtype=np.float32).reshape(-1, 1),
        }

    def __len__(self):
        return len(self.buffer)


def soft_update(target, source, tau=0.005):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_(tau * sp.data + (1 - tau) * tp.data)


print("Guided SAC base components loaded (all ops flatten-safe)")

## 6. Belief-aware Guided SAC (BA-GSAC)

BA-GSAC extends GSAC with an ensemble forward model, whose disagreement measures epistemic uncertainty about the environment state, and adaptive guidance gating, which modulates the distillation coefficient: more guidance under high uncertainty, less when the agent is confident. A VAE belief state is available as an option.

In [ ]:
# Ensemble Forward Model (flatten-safe)

class EnsembleForwardModel(nn.Module):
    def __init__(self, input_dim, action_dim, obs_dim,
                 hidden_dim=128, ensemble_size=5):
        super().__init__()
        self.ensemble_size = ensemble_size
        self.input_total = input_dim + action_dim
        self.models = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.input_total, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, obs_dim),
            ) for _ in range(ensemble_size)
        ])

    def forward(self, partial_hist, action):
        # Ensure 2D
        ph = partial_hist.reshape(partial_hist.shape[0], -1)
        a = action.reshape(action.shape[0], -1)
        x = torch.cat([ph, a], dim=-1)
        return torch.stack([m(x) for m in self.models], dim=1)

    def predict_with_uncertainty(self, partial_hist, action):
        preds = self.forward(partial_hist, action)
        mean_pred = preds.mean(dim=1)
        var_pred = preds.var(dim=1)
        uncertainty = var_pred.mean(dim=-1)
        B, N, D = preds.shape
        diffs = []
        for i in range(N):
            for j in range(i+1, N):
                diffs.append((preds[:,i] - preds[:,j]).pow(2).mean(dim=-1))
        disagreement = torch.stack(diffs, dim=-1).mean(dim=-1)
        return {"mean_pred": mean_pred, "variance": uncertainty,
                "disagreement": disagreement, "all_preds": preds}


# Adaptive Guidance Module

class AdaptiveGuidance:
    def __init__(self, config):
        self.mode = config.get("guidance_mode", "adaptive")
        self.base_coef = config.get("base_distill_coef", 0.1)
        self.min_coef = config.get("min_distill_coef", 0.0)
        self.max_coef = config.get("max_distill_coef", 0.5)
        self.warmup = config.get("guidance_warmup", 2000)
        self.count = 0
        self.warmup_buf = []
        self.ref_lo, self.ref_hi = 0.0, 1.0

    def get_coef(self, uncertainty):
        if self.mode == "fixed":
            return self.base_coef
        self.count += 1
        if self.count <= self.warmup:
            self.warmup_buf.append(uncertainty)
            if self.count == self.warmup:
                arr = np.array(self.warmup_buf)
                self.ref_lo = np.percentile(arr, 10)
                self.ref_hi = np.percentile(arr, 90)
                if self.ref_hi - self.ref_lo < 1e-6:
                    self.ref_hi = self.ref_lo + arr.std() + 1e-6
                self.warmup_buf = []
            return self.base_coef
        rng = self.ref_hi - self.ref_lo
        t = np.clip((uncertainty - self.ref_lo) / rng, 0.0, 1.0)
        if self.mode == "threshold":
            return self.max_coef if t > 0.5 else self.min_coef
        return self.min_coef + (self.max_coef - self.min_coef) * t


print("BA-GSAC components loaded (Ensemble + Adaptive Guidance, flatten-safe)")

## 7. BA-GSAC agent for Highway-Env

In [ ]:
# BA-GSAC Agent for Highway-Env (all tensors guaranteed 2D)

class BAGSACAgent:
    DEFAULT_CONFIG = {
        "hidden_dim": 128, "lr": 3e-4, "distill_lr": 3e-4,
        "gamma": 0.99, "tau": 0.005, "init_alpha": 0.2,
        "alpha_lr": 3e-4, "batch_size": 128, "replay_capacity": 50000,
        "history_len": 3, "random_start": 500,
        "ensemble_size": 5, "ensemble_hidden_dim": 128, "ensemble_lr": 3e-4,
        "guidance_mode": "adaptive", "base_distill_coef": 0.1,
        "min_distill_coef": 0.01, "max_distill_coef": 0.5, "guidance_warmup": 800,
    }

    def __init__(self, obs_dim, action_dim, action_scale=1.0,
                 config=None, device="cpu"):
        self.cfg = {**self.DEFAULT_CONFIG, **(config or {})}
        self.device = device
        self.obs_dim = obs_dim  # flat obs dim
        self.action_dim = action_dim
        self.action_scale = action_scale
        self.hist_dim = obs_dim * self.cfg["history_len"]

        hdim = self.cfg["hidden_dim"]

        # Guiding actor: full state input
        self.guiding_actor = GaussianPolicy(obs_dim, action_dim, hdim, action_scale).to(device)
        # Control actor: history input
        self.control_actor = GaussianPolicy(self.hist_dim, action_dim, hdim, action_scale).to(device)
        # Critic: full state
        self.critic = TwinQNetwork(obs_dim, action_dim, hdim).to(device)
        self.critic_target = deepcopy(self.critic).to(device)
        for p in self.critic_target.parameters():
            p.requires_grad = False
        self.distill_dnn = DistillationDNN(self.hist_dim, action_dim, hdim, action_scale).to(device)

        self.guiding_optim = optim.Adam(self.guiding_actor.parameters(), lr=self.cfg["lr"])
        self.control_optim = optim.Adam(self.control_actor.parameters(), lr=self.cfg["lr"])
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=self.cfg["lr"])
        self.distill_optim = optim.Adam(self.distill_dnn.parameters(), lr=self.cfg["distill_lr"])

        self.target_entropy = -action_dim
        self.log_alpha = torch.tensor(np.log(self.cfg["init_alpha"]),
                                      dtype=torch.float32, requires_grad=True, device=device)
        self.alpha_optim = optim.Adam([self.log_alpha], lr=self.cfg["alpha_lr"])

        # Ensemble
        self.ensemble = EnsembleForwardModel(
            self.hist_dim, action_dim, obs_dim,
            self.cfg["ensemble_hidden_dim"], self.cfg["ensemble_size"]).to(device)
        self.ensemble_optim = optim.Adam(self.ensemble.parameters(), lr=self.cfg["ensemble_lr"])

        self.adaptive_guidance = AdaptiveGuidance(self.cfg)
        self.current_distill_coef = self.cfg["base_distill_coef"]
        self.replay_buffer = ReplayBuffer(self.cfg["replay_capacity"])
        self.train_metrics = {"critic_loss": [], "control_loss": [],
                              "disagreement": [], "distill_coef": [], "alpha": []}

    @property
    def alpha(self):
        return self.log_alpha.exp().detach()

    def _t(self, x):
        return torch.as_tensor(np.asarray(x, dtype=np.float32), device=self.device).reshape(1, -1)

    def _tb(self, x):
        """Batch tensor from numpy array."""
        return torch.as_tensor(np.asarray(x, dtype=np.float32), device=self.device)

    def select_action(self, obs_hist, deterministic=False):
        ht = self._t(obs_hist)
        with torch.no_grad():
            if deterministic:
                _, _, action = self.control_actor.sample(ht)
            else:
                action, _, _ = self.control_actor.sample(ht)
        return action.cpu().numpy().flatten()

    def select_guiding_action(self, full_state):
        st = self._t(full_state)
        with torch.no_grad():
            action, _, _ = self.guiding_actor.sample(st)
        return action.cpu().numpy().flatten()

    def get_uncertainty(self, obs_hist, action):
        ht = self._t(obs_hist)
        at = self._t(action)
        with torch.no_grad():
            out = self.ensemble.predict_with_uncertainty(ht, at)
        return out["disagreement"].item()

    def update(self):
        if len(self.replay_buffer) < self.cfg["batch_size"]:
            return {}

        batch = self.replay_buffer.sample(self.cfg["batch_size"])
        s      = self._tb(batch["full_states"])
        ph     = self._tb(batch["partial_hists"])
        a      = self._tb(batch["actions"])
        r      = self._tb(batch["rewards"])
        s_next = self._tb(batch["next_full_states"])
        ph_next = self._tb(batch["next_partial_hists"])
        d      = self._tb(batch["dones"])

        alpha = self.alpha
        metrics = {}

        # Belief uncertainty
        with torch.no_grad():
            ens_out = self.ensemble.predict_with_uncertainty(ph, a)
            batch_unc = ens_out["disagreement"].mean().item()
        self.current_distill_coef = self.adaptive_guidance.get_coef(batch_unc)
        metrics["disagreement"] = batch_unc
        metrics["distill_coef"] = self.current_distill_coef

        # Critic
        with torch.no_grad():
            na, nlp, _ = self.guiding_actor.sample(s_next)
            q1t, q2t = self.critic_target(s_next, na)
            td_target = r + (1 - d) * self.cfg["gamma"] * (torch.min(q1t, q2t) - alpha * nlp)
        q1, q2 = self.critic(s, a)
        critic_loss = F.mse_loss(q1, td_target) + F.mse_loss(q2, td_target)
        self.critic_optim.zero_grad()
        critic_loss.backward()
        self.critic_optim.step()

        # Guiding actor
        ag, lpg, _ = self.guiding_actor.sample(s)
        q1g, q2g = self.critic(s, ag)
        guiding_loss = (alpha * lpg - torch.min(q1g, q2g)).mean()
        self.guiding_optim.zero_grad()
        guiding_loss.backward()
        self.guiding_optim.step()

        # Control actor (adaptive distillation)
        ac, lpc, _ = self.control_actor.sample(ph)
        q1c, q2c = self.critic(s, ac)
        rl_loss = (alpha * lpc - torch.min(q1c, q2c)).mean()
        with torch.no_grad():
            distill_target = self.distill_dnn(ph)
        control_loss = rl_loss + self.current_distill_coef * F.mse_loss(ac, distill_target)
        self.control_optim.zero_grad()
        control_loss.backward()
        self.control_optim.step()

        # DNN distillation
        with torch.no_grad():
            _, _, guiding_det = self.guiding_actor.sample(s)
        distill_loss = F.mse_loss(self.distill_dnn(ph), guiding_det)
        self.distill_optim.zero_grad()
        distill_loss.backward()
        self.distill_optim.step()

        # Entropy
        with torch.no_grad():
            _, lp_alpha, _ = self.guiding_actor.sample(s)
        alpha_loss = -(self.log_alpha.exp() * (lp_alpha + self.target_entropy)).mean()
        self.alpha_optim.zero_grad()
        alpha_loss.backward()
        self.alpha_optim.step()

        # Target network
        soft_update(self.critic_target, self.critic, self.cfg["tau"])

        # Ensemble
        preds = self.ensemble(ph, a)
        ens_loss = sum(F.mse_loss(preds[:, i], s_next) for i in range(self.cfg["ensemble_size"]))
        ens_loss /= self.cfg["ensemble_size"]
        self.ensemble_optim.zero_grad()
        ens_loss.backward()
        self.ensemble_optim.step()

        metrics["critic_loss"] = critic_loss.item()
        metrics["control_loss"] = control_loss.item()
        metrics["alpha"] = alpha.item()
        for k in self.train_metrics:
            if k in metrics:
                self.train_metrics[k].append(metrics[k])
        return metrics


print("BAGSACAgent ready (flatten-safe for Highway-Env)")

## 7b. Vanilla SAC baseline

Standard SAC on partial observations, without guidance or history. It shows the difficulty of the POMDP setting without any guidance mechanism.

In [ ]:
# Vanilla SAC Agent (POMDP Baseline)

class SACAgent:
    """
    Standard SAC on partial observations, without guidance or history.
    Adapted from the UAI 2026 experimental codebase.
    """
    DEFAULT_CONFIG = {
        "hidden_dim": 128, "lr": 3e-4, "gamma": 0.99, "tau": 0.005,
        "init_alpha": 0.2, "alpha_lr": 3e-4, "batch_size": 128,
        "replay_capacity": 50000, "random_start": 500,
    }

    def __init__(self, obs_dim, action_dim, action_scale=1.0,
                 config=None, device="cpu"):
        self.cfg = {**self.DEFAULT_CONFIG, **(config or {})}
        self.device = device
        self.obs_dim = obs_dim
        self.action_dim = action_dim
        self.action_scale = action_scale

        h = self.cfg["hidden_dim"]
        self.actor = GaussianPolicy(obs_dim, action_dim, h, action_scale).to(device)
        self.critic = TwinQNetwork(obs_dim, action_dim, h).to(device)
        self.critic_target = deepcopy(self.critic).to(device)
        for p in self.critic_target.parameters():
            p.requires_grad = False

        self.actor_optim = optim.Adam(self.actor.parameters(), lr=self.cfg["lr"])
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=self.cfg["lr"])

        self.target_entropy = -action_dim
        self.log_alpha = torch.tensor(
            np.log(self.cfg["init_alpha"]),
            dtype=torch.float32, requires_grad=True, device=device)
        self.alpha_optim = optim.Adam([self.log_alpha], lr=self.cfg["alpha_lr"])

        # Simple replay buffer (no full state needed)
        self.buffer = []
        self.buf_pos = 0
        self.buf_cap = self.cfg["replay_capacity"]

        self.train_metrics = {"critic_loss": []}

    def _t(self, x):
        return torch.as_tensor(x, dtype=torch.float32, device=self.device)

    def _push(self, s, a, r, ns, d):
        data = (s, a, r, ns, float(d))
        if len(self.buffer) < self.buf_cap:
            self.buffer.append(data)
        else:
            self.buffer[self.buf_pos] = data
        self.buf_pos = (self.buf_pos + 1) % self.buf_cap

    def _sample(self, bs):
        idx = np.random.randint(0, len(self.buffer), bs)
        b = [self.buffer[i] for i in idx]
        s, a, r, ns, d = zip(*b)
        return (self._t(np.array(s)), self._t(np.array(a)),
                self._t(np.array(r).reshape(-1,1)),
                self._t(np.array(ns)), self._t(np.array(d).reshape(-1,1)))

    def select_action(self, obs, deterministic=False):
        st = self._t(obs).unsqueeze(0)
        with torch.no_grad():
            if deterministic:
                _, _, action = self.actor.sample(st)
            else:
                action, _, _ = self.actor.sample(st)
        return action.cpu().numpy().flatten()

    def update(self):
        if len(self.buffer) < self.cfg["batch_size"]:
            return {}
        s, a, r, ns, d = self._sample(self.cfg["batch_size"])
        alpha = self.log_alpha.exp().detach()

        # Critic
        with torch.no_grad():
            na, nlp, _ = self.actor.sample(ns)
            q1t, q2t = self.critic_target(ns, na)
            td = r + (1-d) * self.cfg["gamma"] * (torch.min(q1t, q2t) - alpha * nlp)
        q1, q2 = self.critic(s, a)
        cl = F.mse_loss(q1, td) + F.mse_loss(q2, td)
        self.critic_optim.zero_grad()
        cl.backward()
        self.critic_optim.step()

        # Actor
        na2, lp2, _ = self.actor.sample(s)
        q1a, q2a = self.critic(s, na2)
        al = (alpha * lp2 - torch.min(q1a, q2a)).mean()
        self.actor_optim.zero_grad()
        al.backward()
        self.actor_optim.step()

        # Alpha
        with torch.no_grad():
            _, lpa, _ = self.actor.sample(s)
        aloss = -(self.log_alpha.exp() * (lpa + self.target_entropy)).mean()
        self.alpha_optim.zero_grad()
        aloss.backward()
        self.alpha_optim.step()

        soft_update(self.critic_target, self.critic, self.cfg["tau"])
        self.train_metrics["critic_loss"].append(cl.item())
        return {"critic_loss": cl.item()}


print("Vanilla SAC baseline loaded")

## 7c. Guided SAC with fixed guidance

Standard Guided SAC with a constant distillation coefficient (Haklıdır and Temeltaş, 2021). Same architecture as BA-GSAC; this ablation isolates the effect of adaptive guidance.

In [ ]:
# GSAC Agent with Fixed Guidance (Paper Baseline)

class GSACAgent(BAGSACAgent):
    """
    Guided SAC with fixed distillation coefficient.
    Inherits BA-GSAC architecture but disables adaptive guidance.
    This is equivalent to the original Guided SAC (Haklidir & Temeltas, 2021).
    """
    def __init__(self, obs_dim, action_dim, action_scale=1.0,
                 config=None, device="cpu"):
        fixed_cfg = {**(config or {}), "guidance_mode": "fixed"}
        super().__init__(obs_dim, action_dim, action_scale, fixed_cfg, device)


print("GSAC (fixed guidance) baseline loaded")

## 8. Training loop

Iterative interaction: the guiding and control actors alternate steps with the environment, and the control actor sees only partial observations. The default parameters fit a Colab session of 20 to 30 minutes: 15,000 steps per experiment, three POMDP levels (mild, moderate, severe), compared with the fixed-guidance baseline.

In [ ]:
# Training Function (flatten-safe)

def train_agent(env_id="highway-v0", env_config=None, pomdp_level="moderate",
                guidance_mode="adaptive", total_steps=15000,
                eval_interval=1000, eval_episodes=5, seed=42, verbose=True):
    np.random.seed(seed)
    torch.manual_seed(seed)

    cfg = env_config or HIGHWAY_CONFIG
    base_env = make_env(env_id, cfg)
    env = POMDPWrapper(base_env, pomdp_level=pomdp_level)

    obs_dim = env.flat_obs_dim  # guaranteed flat
    act_dim = env.action_space.shape[0]
    act_scale = float(env.action_space.high[0])

    agent_cfg = {"guidance_mode": guidance_mode, "random_start": 500,
                 "history_len": 3, "batch_size": 128, "replay_capacity": 50000,
                 "guidance_warmup": 800, "hidden_dim": 128}
    agent = BAGSACAgent(obs_dim, act_dim, act_scale, agent_cfg, device)

    hist = ObservationHistory(obs_dim, agent.cfg["history_len"])

    eval_returns = []
    eval_uncertainties = []
    ep_count = 0
    current_ep_reward = 0

    partial_obs, info = env.reset(seed=seed)
    full_obs = info["full_obs"]
    hist.reset(partial_obs)

    step_bar = tqdm(range(total_steps),
                    desc=f"Training [{pomdp_level}/{guidance_mode}]",
                    disable=not verbose)

    for step in step_bar:
        current_hist = hist.get()

        if step < agent.cfg["random_start"]:
            action = env.action_space.sample()
        elif step % 2 == 0:
            action = agent.select_guiding_action(full_obs)
        else:
            action = agent.select_action(current_hist)

        action = np.clip(action, -act_scale, act_scale)
        next_partial, reward, terminated, truncated, info = env.step(action)
        next_full = info["full_obs"]
        done = terminated or truncated

        hist.push(next_partial)
        next_hist = hist.get()

        agent.replay_buffer.push(full_obs, current_hist, action, reward,
                                 next_full, next_hist, done)
        current_ep_reward += reward

        if done:
            ep_count += 1
            current_ep_reward = 0
            partial_obs, info = env.reset()
            full_obs = info["full_obs"]
            hist.reset(partial_obs)
        else:
            partial_obs = next_partial
            full_obs = next_full

        if step >= agent.cfg["random_start"]:
            agent.update()

        if (step + 1) % eval_interval == 0:
            eval_ret, eval_unc = evaluate_agent(
                agent, env_id, cfg, pomdp_level, eval_episodes, seed_offset=step)
            eval_returns.append(eval_ret)
            eval_uncertainties.append(eval_unc)
            if verbose:
                step_bar.set_postfix({"ret": f"{eval_ret:.1f}",
                                      "unc": f"{eval_unc:.4f}",
                                      "coef": f"{agent.current_distill_coef:.3f}"})

    env.close()
    return agent, eval_returns, eval_uncertainties, agent.train_metrics


def evaluate_agent(agent, env_id, env_config, pomdp_level,
                   num_episodes=5, seed_offset=0):
    base_env = make_env(env_id, env_config)
    env = POMDPWrapper(base_env, pomdp_level=pomdp_level)
    hist = ObservationHistory(agent.obs_dim, agent.cfg["history_len"])

    returns = []
    uncertainties = []

    for ep in range(num_episodes):
        partial_obs, info = env.reset(seed=seed_offset + ep)
        hist.reset(partial_obs)
        ret, done = 0.0, False

        while not done:
            current_hist = hist.get()
            action = agent.select_action(current_hist, deterministic=True)
            action = np.clip(action, -agent.action_scale, agent.action_scale)
            unc = agent.get_uncertainty(current_hist, action)
            uncertainties.append(unc)
            next_partial, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            ret += reward
            hist.push(next_partial)
        returns.append(ret)

    env.close()
    return np.mean(returns), np.mean(uncertainties) if uncertainties else 0.0


print("Training loop ready (flatten-safe)")

## Experiment setup and checkpoints

In [ ]:
import json, os, zipfile, time

SAVE_DIR = '/content/reviewer_results'
os.makedirs(SAVE_DIR, exist_ok=True)

ALL_RESULTS = {}  # master results dict

TOTAL_STEPS = 50000
EVAL_INTERVAL = 1500
SEEDS = [42, 123, 7]

def save_checkpoint(name):
    """Save current results to disk after each experiment block."""
    with open(f'{SAVE_DIR}/{name}.json', 'w') as f:
        json.dump(ALL_RESULTS, f, indent=2, default=str)
    print(f'Checkpoint saved: {name} ({len(ALL_RESULTS)} experiments)')

def train_custom_cfg(pomdp_level, seed, custom_agent_cfg, label='custom'):
    """Train with custom agent config (for λ ablations)."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    cfg = HIGHWAY_CONFIG
    base_env = make_env('highway-v0', cfg)
    env = POMDPWrapper(base_env, pomdp_level=pomdp_level)
    obs_dim = env.flat_obs_dim
    act_dim = env.action_space.shape[0]
    act_scale = float(env.action_space.high[0])
    agent = BAGSACAgent(obs_dim, act_dim, act_scale, custom_agent_cfg, device)
    hist = ObservationHistory(obs_dim, agent.cfg['history_len'])
    partial_obs, info = env.reset(seed=seed)
    full_obs = info['full_obs']
    hist.reset(partial_obs)
    evals_list = []
    collisions = 0
    episodes = 0
    for step in tqdm(range(TOTAL_STEPS), desc=f'{label} [s{seed}]'):
        current_hist = hist.get()
        if step < agent.cfg['random_start']:
            action = env.action_space.sample()
        elif step % 2 == 0:
            action = agent.select_guiding_action(full_obs)
        else:
            action = agent.select_action(current_hist)
        action = np.clip(action, -act_scale, act_scale)
        next_partial, reward, terminated, truncated, info = env.step(action)
        next_full = info['full_obs']
        done = terminated or truncated
        hist.push(next_partial)
        next_hist = hist.get()
        agent.replay_buffer.push(full_obs, current_hist, action, reward,
                                 next_full, next_hist, done)
        if done:
            episodes += 1
            if terminated:  # collision
                collisions += 1
            partial_obs, info = env.reset()
            full_obs = info['full_obs']
            hist.reset(partial_obs)
        else:
            partial_obs = next_partial
            full_obs = next_full
        if step >= agent.cfg['random_start']:
            agent.update()
        if (step + 1) % EVAL_INTERVAL == 0:
            eval_ret, eval_unc = evaluate_agent(
                agent, 'highway-v0', cfg, pomdp_level, 5, seed_offset=step)
            evals_list.append(eval_ret)
    env.close()
    collision_rate = collisions / max(episodes, 1)
    return evals_list, collision_rate, episodes

def train_sac_with_history(pomdp_level, seed, history_len=3):
    """SAC with stacked history (fair POMDP baseline)."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    cfg = HIGHWAY_CONFIG
    base_env = make_env('highway-v0', cfg)
    env = POMDPWrapper(base_env, pomdp_level=pomdp_level)
    obs_dim = env.flat_obs_dim
    act_dim = env.action_space.shape[0]
    act_scale = float(env.action_space.high[0])
    hist_dim = obs_dim * history_len
    sac = SACAgent(hist_dim, act_dim, act_scale, device=device)
    obs_hist = ObservationHistory(obs_dim, history_len)
    partial_obs, info = env.reset(seed=seed)
    obs_hist.reset(partial_obs)
    evals_list = []
    collisions = 0
    episodes = 0
    for step in tqdm(range(TOTAL_STEPS), desc=f'SAC-hist [s{seed}]'):
        current_hist = obs_hist.get()
        if step < sac.cfg['random_start']:
            action = env.action_space.sample()
        else:
            action = sac.select_action(current_hist)
        action = np.clip(action, -act_scale, act_scale)
        next_partial, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        obs_hist.push(next_partial)
        next_hist_arr = obs_hist.get()
        sac._push(current_hist, action, reward, next_hist_arr, done)
        if done:
            episodes += 1
            if terminated:
                collisions += 1
            partial_obs, info = env.reset()
            obs_hist.reset(partial_obs)
        else:
            partial_obs = next_partial
        if step >= sac.cfg['random_start']:
            sac.update()
        if (step + 1) % EVAL_INTERVAL == 0:
            eval_env = POMDPWrapper(make_env('highway-v0', cfg), pomdp_level)
            eval_hist = ObservationHistory(obs_dim, history_len)
            er_list = []
            for ep in range(5):
                eo, _ = eval_env.reset(seed=step+ep)
                eval_hist.reset(eo)
                er, ed = 0, False
                while not ed:
                    eh = eval_hist.get()
                    ea = sac.select_action(eh, deterministic=True)
                    eo, r, te, tr, _ = eval_env.step(np.clip(ea, -act_scale, act_scale))
                    eval_hist.push(eo)
                    ed = te or tr
                    er += r
                er_list.append(er)
            eval_env.close()
            evals_list.append(np.mean(er_list))
    env.close()
    collision_rate = collisions / max(episodes, 1)
    return evals_list, collision_rate, episodes

def train_vanilla_sac(pomdp_level, seed):
    """Vanilla SAC (no history, no guidance) with collision tracking."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    cfg = HIGHWAY_CONFIG
    base_env = make_env('highway-v0', cfg)
    env = POMDPWrapper(base_env, pomdp_level=pomdp_level)
    obs_dim = env.flat_obs_dim
    act_dim = env.action_space.shape[0]
    act_scale = float(env.action_space.high[0])
    sac = SACAgent(obs_dim, act_dim, act_scale, device=device)
    partial_obs, info = env.reset(seed=seed)
    evals_list = []
    collisions = 0
    episodes = 0
    for step in tqdm(range(TOTAL_STEPS), desc=f'SAC [s{seed}]'):
        if step < sac.cfg['random_start']:
            action = env.action_space.sample()
        else:
            action = sac.select_action(partial_obs)
        action = np.clip(action, -act_scale, act_scale)
        next_partial, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        sac._push(partial_obs, action, reward, next_partial, done)
        if done:
            episodes += 1
            if terminated:
                collisions += 1
            partial_obs, info = env.reset()
        else:
            partial_obs = next_partial
        if step >= sac.cfg['random_start']:
            sac.update()
        if (step + 1) % EVAL_INTERVAL == 0:
            eval_env = POMDPWrapper(make_env('highway-v0', cfg), pomdp_level)
            er_list = []
            for ep in range(5):
                eo, _ = eval_env.reset(seed=step+ep)
                er, ed = 0, False
                while not ed:
                    ea = sac.select_action(eo, deterministic=True)
                    eo, r, te, tr, _ = eval_env.step(np.clip(ea, -act_scale, act_scale))
                    ed = te or tr
                    er += r
                er_list.append(er)
            eval_env.close()
            evals_list.append(np.mean(er_list))
    env.close()
    collision_rate = collisions / max(episodes, 1)
    return evals_list, collision_rate, episodes

print('Experiment infrastructure ready')
print(f'   Seeds: {SEEDS}')
print(f'   Steps: {TOTAL_STEPS}')
print(f'   Eval interval: {EVAL_INTERVAL}')


## Exp 1: Fixed λ=0.01 Baseline (ALL POMDP levels, 3 seeds)
Reviewer 1's #1 concern: Is BA-GSAC better because of *adaptive* guidance or just *less* guidance?

~9 runs × 45min = ~6.75 hours

In [ ]:
# EXP 1: Fixed λ=0.01. ALL POMDP levels × 3 seeds
cfg_lam001 = {
    'guidance_mode': 'fixed', 'base_distill_coef': 0.01,
    'random_start': 500, 'history_len': 3,
    'batch_size': 128, 'replay_capacity': 50000,
    'guidance_warmup': 800, 'hidden_dim': 128,
}

for pomdp in ['mild', 'moderate', 'severe']:
    for seed in SEEDS:
        key = f'{pomdp}_fixed001_s{seed}'
        print(f'\n{"="*50}')
        print(f'  Fixed λ=0.01 | {pomdp} | seed={seed}')
        print(f'{"="*50}')
        evals, coll_rate, eps = train_custom_cfg(pomdp, seed, cfg_lam001, f'λ=0.01-{pomdp}')
        ALL_RESULTS[key] = {'eval_returns': evals, 'collision_rate': coll_rate, 'episodes': eps}
        last5 = evals[-5:]
        print(f'  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}  coll={coll_rate:.2%}')

save_checkpoint('exp1_fixed_lam001')


## Exp 2: Fixed λ=0.5 Baseline (severe only, 3 seeds)
Control experiment: does heavy guidance help under severe POMDP?

~3 runs × 45min = ~2.25 hours

In [ ]:
# EXP 2: Fixed λ=0.5, severe × 3 seeds
cfg_lam05 = {
    'guidance_mode': 'fixed', 'base_distill_coef': 0.5,
    'random_start': 500, 'history_len': 3,
    'batch_size': 128, 'replay_capacity': 50000,
    'guidance_warmup': 800, 'hidden_dim': 128,
}

for seed in SEEDS:
    key = f'severe_fixed05_s{seed}'
    print(f'\n{"="*50}')
    print(f'  Fixed λ=0.5 | severe | seed={seed}')
    print(f'{"="*50}')
    evals, coll_rate, eps = train_custom_cfg('severe', seed, cfg_lam05, 'λ=0.5')
    ALL_RESULTS[key] = {'eval_returns': evals, 'collision_rate': coll_rate, 'episodes': eps}
    last5 = evals[-5:]
    print(f'  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}  coll={coll_rate:.2%}')

save_checkpoint('exp2_fixed_lam05')


## Exp 3: Linear Decay λ Schedule (severe, 3 seeds)
Reviewer 1: "Would a simple decay schedule work as well?"

λ decays linearly: 0.1 to 0.01 over training.

~3 runs × 45min = ~2.25 hours

In [ ]:
# EXP 3: Linear Decay λ, severe × 3 seeds
# Uses train_agent with fixed mode, then manually decays
for seed in SEEDS:
    key = f'severe_decay_s{seed}'
    print(f'\n{"="*50}')
    print(f'  Linear Decay λ | severe | seed={seed}')
    print(f'{"="*50}')

    np.random.seed(seed)
    torch.manual_seed(seed)
    cfg = HIGHWAY_CONFIG
    base_env = make_env('highway-v0', cfg)
    env = POMDPWrapper(base_env, pomdp_level='severe')
    obs_dim = env.flat_obs_dim
    act_dim = env.action_space.shape[0]
    act_scale = float(env.action_space.high[0])

    agent_cfg = {'guidance_mode': 'fixed', 'base_distill_coef': 0.1,
                 'random_start': 500, 'history_len': 3,
                 'batch_size': 128, 'replay_capacity': 50000,
                 'guidance_warmup': 800, 'hidden_dim': 128}
    agent = BAGSACAgent(obs_dim, act_dim, act_scale, agent_cfg, device)
    hist = ObservationHistory(obs_dim, agent.cfg['history_len'])
    partial_obs, info = env.reset(seed=seed)
    full_obs = info['full_obs']
    hist.reset(partial_obs)

    evals_list = []
    collisions = 0
    episodes = 0
    for step in tqdm(range(TOTAL_STEPS), desc=f'decay [s{seed}]'):
        # Linear decay: 0.1 to 0.01
        progress = step / TOTAL_STEPS
        agent.current_distill_coef = 0.1 * (1 - progress) + 0.01 * progress

        current_hist = hist.get()
        if step < agent.cfg['random_start']:
            action = env.action_space.sample()
        elif step % 2 == 0:
            action = agent.select_guiding_action(full_obs)
        else:
            action = agent.select_action(current_hist)
        action = np.clip(action, -act_scale, act_scale)
        next_partial, reward, terminated, truncated, info = env.step(action)
        next_full = info['full_obs']
        done = terminated or truncated
        hist.push(next_partial)
        next_hist = hist.get()
        agent.replay_buffer.push(full_obs, current_hist, action, reward,
                                 next_full, next_hist, done)
        if done:
            episodes += 1
            if terminated: collisions += 1
            partial_obs, info = env.reset()
            full_obs = info['full_obs']
            hist.reset(partial_obs)
        else:
            partial_obs = next_partial
            full_obs = next_full
        if step >= agent.cfg['random_start']:
            agent.update()
        if (step + 1) % EVAL_INTERVAL == 0:
            eval_ret, eval_unc = evaluate_agent(
                agent, 'highway-v0', cfg, 'severe', 5, seed_offset=step)
            evals_list.append(eval_ret)
    env.close()
    coll_rate = collisions / max(episodes, 1)
    ALL_RESULTS[key] = {'eval_returns': evals_list, 'collision_rate': coll_rate, 'episodes': episodes}
    last5 = evals_list[-5:]
    print(f'  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}  coll={coll_rate:.2%}')

save_checkpoint('exp3_decay')


## Exp 4: SAC + History K=3 (ALL POMDP levels, 3 seeds)
Reviewer 3's #1 concern: Vanilla SAC has no history, unfair comparison.

~9 runs × 45min = ~6.75 hours

In [ ]:
# EXP 4: SAC + History K=3. ALL POMDP × 3 seeds
for pomdp in ['mild', 'moderate', 'severe']:
    for seed in SEEDS:
        key = f'{pomdp}_sachist_s{seed}'
        print(f'\n{"="*50}')
        print(f'  SAC+History | {pomdp} | seed={seed}')
        print(f'{"="*50}')
        evals, coll_rate, eps = train_sac_with_history(pomdp, seed, history_len=3)
        ALL_RESULTS[key] = {'eval_returns': evals, 'collision_rate': coll_rate, 'episodes': eps}
        last5 = evals[-5:]
        print(f'  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}  coll={coll_rate:.2%}')

save_checkpoint('exp4_sac_history')


## Exp 5: Multi-Seed for Mild & Moderate (BA-GSAC, GSAC, SAC)
All 3 reviewers: Single-seed for mild/moderate is insufficient.

~18 runs × 45min = ~13.5 hours

Note: This overlaps with Exp 1 (fixed λ=0.01) for mild/moderate, but we need BA-GSAC and GSAC too.

In [ ]:
# EXP 5: Multi-seed mild/moderate. BA-GSAC, GSAC, SAC
for pomdp in ['mild', 'moderate']:
    # BA-GSAC (adaptive)
    for seed in SEEDS:
        key = f'{pomdp}_bagsac_s{seed}'
        if key in ALL_RESULTS:
            print(f'  Skipping {key} (already computed)')
            continue
        print(f'\n{"="*50}')
        print(f'  BA-GSAC | {pomdp} | seed={seed}')
        print(f'{"="*50}')
        agent, evals, uncs, metrics = train_agent(
            env_id='highway-v0', env_config=HIGHWAY_CONFIG,
            pomdp_level=pomdp, guidance_mode='adaptive',
            total_steps=TOTAL_STEPS, eval_interval=EVAL_INTERVAL, seed=seed)
        ALL_RESULTS[key] = {'eval_returns': evals, 'eval_uncertainties': uncs}
        last5 = evals[-5:]
        print(f'  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}')

    # GSAC (fixed λ=0.1)
    for seed in SEEDS:
        key = f'{pomdp}_gsac_s{seed}'
        if key in ALL_RESULTS:
            print(f'  Skipping {key} (already computed)')
            continue
        print(f'\n{"="*50}')
        print(f'  GSAC | {pomdp} | seed={seed}')
        print(f'{"="*50}')
        agent, evals, uncs, metrics = train_agent(
            env_id='highway-v0', env_config=HIGHWAY_CONFIG,
            pomdp_level=pomdp, guidance_mode='fixed',
            total_steps=TOTAL_STEPS, eval_interval=EVAL_INTERVAL, seed=seed)
        ALL_RESULTS[key] = {'eval_returns': evals, 'eval_uncertainties': uncs}
        last5 = evals[-5:]
        print(f'  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}')

    # Vanilla SAC
    for seed in SEEDS:
        key = f'{pomdp}_sac_s{seed}'
        if key in ALL_RESULTS:
            print(f'  Skipping {key} (already computed)')
            continue
        print(f'\n{"="*50}')
        print(f'  Vanilla SAC | {pomdp} | seed={seed}')
        print(f'{"="*50}')
        evals, coll_rate, eps = train_vanilla_sac(pomdp, seed)
        ALL_RESULTS[key] = {'eval_returns': evals, 'collision_rate': coll_rate}
        last5 = evals[-5:]
        print(f'  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}')

    save_checkpoint(f'exp5_{pomdp}_multiseed')


In [ ]:
# Moderate POMDP multi-seed (GSAC + SAC only)
TOTAL_STEPS = 50000
EVAL_INTERVAL = 1500

for seed in [42, 123, 7]:
    # GSAC λ=0.1
    print(f"\n{'='*50}")
    print(f"  GSAC λ=0.1 | moderate | seed={seed}")
    print(f"{'='*50}")
    agent, evals, uncs, metrics = train_agent(
        env_id='highway-v0', env_config=HIGHWAY_CONFIG,
        pomdp_level='moderate', guidance_mode='fixed',
        total_steps=TOTAL_STEPS, eval_interval=EVAL_INTERVAL, seed=seed)
    last5 = evals[-5:]
    print(f"  → Last-5: {np.mean(last5):.1f} ± {np.std(last5):.1f}")

    # Vanilla SAC
    print(f"\n{'='*50}")
    print(f"  Vanilla SAC | moderate | seed={seed}")
    print(f"{'='*50}")
    # (use train_vanilla_sac from reviewer notebook, or the SAC loop from train_agent)

## Summary and download

In [ ]:
# FINAL SUMMARY
print('\n' + '='*70)
print('COMPLETE RESULTS SUMMARY')
print('='*70)

# Helper
def summarize(prefix, name):
    for pomdp in ['mild', 'moderate', 'severe']:
        vals = []
        for seed in SEEDS:
            key = f'{pomdp}_{prefix}_s{seed}'
            if key in ALL_RESULTS:
                evals = ALL_RESULTS[key]['eval_returns']
                vals.append(np.mean(evals[-5:]))
        if vals:
            print(f'  {name:25s} [{pomdp:8s}]: {np.mean(vals):6.1f} ± {np.std(vals):5.1f}  (n={len(vals)} seeds)')

print('\n--- Main Methods ---')
summarize('bagsac', 'BA-GSAC (adaptive)')
summarize('gsac', 'GSAC (fixed λ=0.1)')
summarize('sac', 'Vanilla SAC')

print('\n--- Reviewer Baselines ---')
summarize('fixed001', 'Fixed λ=0.01')
summarize('fixed05', 'Fixed λ=0.5')
summarize('decay', 'Linear Decay')
summarize('sachist', 'SAC + History K=3')

# Collision rates
print('\n--- Collision Rates (severe, 3-seed avg) ---')
for prefix, name in [('bagsac','BA-GSAC'), ('gsac','GSAC'), ('sac','SAC'),
                      ('fixed001','λ=0.01'), ('sachist','SAC+hist')]:
    rates = []
    for seed in SEEDS:
        key = f'severe_{prefix}_s{seed}'
        if key in ALL_RESULTS and 'collision_rate' in ALL_RESULTS[key]:
            rates.append(ALL_RESULTS[key]['collision_rate'])
    if rates:
        print(f'  {name:20s}: {np.mean(rates):.1%}')

# Save final + zip + download
save_checkpoint('FINAL_all_results')

zip_path = '/content/reviewer_experiments_complete.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fname in os.listdir(SAVE_DIR):
        zf.write(f'{SAVE_DIR}/{fname}', fname)

print(f'\n All done! Downloading {zip_path}')
from google.colab import files
files.download(zip_path)
